# 🌤️ Weather Data Preprocessing

This notebook ingests raw hourly weather data for Toronto, cleans and normalises it, and writes it through two pipeline layers:

| Layer | Path | Description |
|---|---|---|
| **Bronze** | `data/bronze/weather` | Raw data as-is, converted to Parquet |
| **Silver** | `data/silver/weather_hourly` | Cleaned, filtered, query-ready |

| Step | Purpose |
|---|---|
| 0 | Define file paths |
| 1 | Read raw CSV with pandas |
| 2 | Rename temperature columns |
| 3 | Normalise the hour column |
| 4 | Convert to Spark DataFrame |
| 5 | Write Bronze layer |
| 6 | Clean and write Silver layer |
| 7 | Validate row counts |


## Step 0–7 — Ingest, Clean & Write Bronze/Silver

This cell runs the full preprocessing pipeline in a single pass:

**Step 0 — Paths:** defines the local CSV source and the DBFS destinations for Bronze and Silver layers.

**Step 1 — Read CSV:** loads the raw weather file using pandas with semicolon delimiter and UTF-8 encoding. An encoding artifact (`Â`) is stripped from column names, which can appear when files are saved with a BOM in Excel.

**Step 2 — Rename temperature columns:** standardises column names regardless of how they were originally exported:
- `temperature_2m_*` → `temperature_2m_celsius`
- `apparent_temperature_*` → `apparent_temperature_celsius`

**Step 3 — Normalise hour:** converts raw hour values (which may come in as `"1"`, `"01"`, `"1:00"`, etc.) into a consistent `HH:00` format (e.g. `"09:00"`). Invalid or missing values are set to `None`.

**Step 4 — Convert to Spark:** converts the cleaned pandas DataFrame into a Spark DataFrame for scalable distributed processing and DBFS writes.

**Step 5 — Write Bronze:** saves the Spark DataFrame to DBFS as Parquet. Bronze is a raw-faithful copy — no rows are dropped at this stage.

**Step 6 — Silver cleaning:** filters out rows with:
- Null `year`
- `month` outside 1–12
- `day` outside 1–31
- Null `hour`

The cleaned result is written to the Silver Parquet path.

**Step 7 — Validation:** prints Bronze and Silver row counts to confirm data was written and quantify how many rows were dropped during cleaning.


In [0]:
import pandas as pd
import re
from pyspark.sql import SparkSession

# ---------------------------
# 0) Paths
# ---------------------------
CSV_LOCAL = "../../data/raw/Toronto_weather_data.csv"

BRONZE_PATH = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze/weather"
SILVER_PATH = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly"

# ---------------------------
# 1) Read CSV with pandas
# ---------------------------
df_raw = pd.read_csv(CSV_LOCAL, sep=";", encoding="utf-8-sig")

print("Rows:", len(df_raw))
print(df_raw.head())

# Fix encoding artifact
df_raw.columns = [c.replace("Â", "").strip() for c in df_raw.columns]

# ---------------------------
# 2) Rename temperature cols safely
# ---------------------------
for c in df_raw.columns:
    if "temperature_2m" in c and "apparent" not in c:
        df_raw = df_raw.rename(columns={c: "temperature_2m_celsius"})
    if "apparent_temperature" in c:
        df_raw = df_raw.rename(columns={c: "apparent_temperature_celsius"})

# ---------------------------
# 3) Normalize hour
# ---------------------------
def normalize_hour(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    m = re.match(r"^(\d{1,2})", s)
    if not m:
        return None
    h = int(m.group(1))
    if 0 <= h <= 23:
        return f"{h:02d}:00"
    return None

df_raw["hour"] = df_raw["hour"].apply(normalize_hour)

# ---------------------------
# 4) Convert to Spark DF
# ---------------------------
spark_df = spark.createDataFrame(df_raw)

# ---------------------------
# 5) Write Bronze
# ---------------------------
spark_df.write.mode("overwrite").parquet(BRONZE_PATH)

print("✅ Bronze written:", BRONZE_PATH)

# ---------------------------
# 6) Basic Silver cleaning
# ---------------------------
from pyspark.sql import functions as F

silver_df = (
    spark_df
    .filter(
        F.col("year").isNotNull() &
        F.col("month").between(1,12) &
        F.col("day").between(1,31) &
        F.col("hour").isNotNull()
    )
)

silver_df.write.mode("overwrite").parquet(SILVER_PATH)

print("✅ Silver written:", SILVER_PATH)

# ---------------------------
# 7) Validation
# ---------------------------
print("Bronze rows:", spark.read.parquet(BRONZE_PATH).count())
print("Silver rows:", spark.read.parquet(SILVER_PATH).count())

display(silver_df.limit(10))


## Hour Distribution Check

Reads the Silver weather table and counts how many records exist per unique `hour` value.

This validates that the hour normalisation in Step 3 worked correctly — every value should follow the `HH:00` format and the distribution should be roughly even across 24 hours. Any unexpected values (e.g. `null`, raw integers, or duplicate formats) will surface here.


In [0]:
weather = spark.read.parquet("dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly")

display(
  weather.select("hour").groupBy("hour").count().orderBy("hour").limit(30)
)


## Null Value Audit

Counts null values across the four core time dimension columns: `year`, `month`, `day`, and `hour`.

All four should show **zero nulls** in the Silver layer, since the filter in Step 6 explicitly removes rows where any of these are missing or invalid. A non-zero count here would indicate a gap in the cleaning logic that needs to be addressed before downstream joins.


In [0]:
from pyspark.sql import functions as F

display(
  weather.select(
    F.count(F.when(F.col("year").isNull(), True)).alias("null_year"),
    F.count(F.when(F.col("month").isNull(), True)).alias("null_month"),
    F.count(F.when(F.col("day").isNull(), True)).alias("null_day"),
    F.count(F.when(F.col("hour").isNull(), True)).alias("null_hour")
  )
)


## Monthly Record Coverage

Groups the Silver data by `year` and `month` and counts the records in each period.

This confirms temporal coverage across the full dataset — each year/month combination should have a consistent record count (~720 hours for a standard month). Months with significantly fewer records may indicate missing data from the source CSV or dropped rows during cleaning, and should be investigated before model training.


In [0]:
display(
  weather.groupBy("year","month").count().orderBy("year","month")
)
